# AI Financial Life Coach (AFILC) — Cuaderno de Modelado Reproducible
### Asignatura 7 · Modelado Predictivo con Machine Learning · Grupo 05
### Máster en Big Data & Business Intelligence — Next Educación / Universidad Isabel I de Burgos

**Integrantes:** C. Solis Meza · J. Cáceres Mondragón · M. Merola · M. Á. Lozano Torres

---

Este cuaderno reproduce la fase de **modelado** del proyecto AFILC sobre el dataset sintético de
estructura Tink (`data/clean/dataset_final_usuarios_tink.csv`: 639.433 registros usuario-mes de
40.000 usuarios). La memoria del proyecto desarrolla la justificación y la interpretación de cada
resultado; este cuaderno ejecuta el código y muestra sus salidas.

## Correspondencia con la rúbrica

| Apartado | Peso | Sección |
|---|---|---|
| 4.1 Preprocesamiento de datos | 20% | §1 |
| 4.2 Modelos predictivos principales | 70% | §2 (Modelo 1 regresión, Modelo 2 clasificación) |
| 4.3 Modelo avanzado (ensemble) | 5% | §3 (+ §4 modelo temporal) |
| 4.4 Análisis no supervisado | 5% | §5 |

## Reproducibilidad

- El dataset de entrada lo genera el pipeline `01_pipeline_tink/06_prepara_usuarios_tink.py` a partir
  de transacciones sintéticas de varios GB que no se versionan en el repositorio; por eso el cuaderno
  parte del dataset ya preparado y **no** reejecuta ese pipeline.
- Cada celda de código ejecuta los scripts de producción (`04_modelos_predictivos/*.py`), los mismos
  que generan las figuras y métricas de la memoria: no hay lógica duplicada.
- Validación: partición train/test por usuario y ajuste de hiperparámetros con GridSearchCV y
  validación cruzada agrupada por usuario (GroupKFold, 5 pliegues).
- Tiempo de ejecución orientativo: entre 5 y 15 minutos, según el equipo (la búsqueda de
  hiperparámetros es la parte más costosa).

**Requisitos:** Python 3.10+, `pandas`, `numpy`, `scikit-learn`, `scipy`, `matplotlib`, `seaborn`.
Ejecutar desde la raíz del repositorio.


In [ ]:
import os, sys, json, subprocess
import pandas as pd
from IPython.display import Image, display

ROOT = os.getcwd()
DATA_CLEAN = os.path.join(ROOT, "data", "clean")
MODELOS_DIR = os.path.join(ROOT, "04_modelos_predictivos")
assert os.path.exists(os.path.join(DATA_CLEAN, "dataset_final_usuarios_tink.csv")), (
    "No se encuentra data/clean/dataset_final_usuarios_tink.csv: ejecutar el cuaderno desde la raíz "
    "del repositorio, con el dataset generado por 01_pipeline_tink/06_prepara_usuarios_tink.py.")

def ejecutar_script(nombre_script, timeout=3600):
    """Ejecuta un script de producción como subproceso (igual que desde la línea de comandos),
    muestra su salida y detiene el cuaderno si el script falla."""
    ruta = os.path.join(MODELOS_DIR, nombre_script)
    print(f">>> Ejecutando {ruta}\n")
    r = subprocess.run([sys.executable, "-W", "ignore", ruta], cwd=MODELOS_DIR,
                       capture_output=True, text=True, timeout=timeout)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{nombre_script} terminó con código {r.returncode}")

def mostrar(png, ancho=850):
    display(Image(filename=os.path.join(DATA_CLEAN, png), width=ancho))

def tabla(registros):
    return pd.DataFrame(registros)


## 1. Preprocesamiento de datos (rúbrica 4.1)

`preprocesamiento.py` aísla tres decisiones, cada una respaldada por un diagnóstico cuantitativo:

1. **Codificación de `perfil`** (junior / medio / senior / freelance) mediante one-hot encoding
   (`drop_first=True`), no ordinal: es una variable nominal sin orden natural.
2. **Asimetría y transformación logarítmica**: log1p en las variables monetarias con |skew| > 1.
3. **Escalado** con StandardScaler ajustado solo sobre entrenamiento (y, en la validación cruzada,
   dentro de cada pliegue mediante un `Pipeline`).


In [ ]:
ejecutar_script("preprocesamiento.py")

In [ ]:
with open(os.path.join(DATA_CLEAN, "preprocesamiento_diagnostico.json"), encoding="utf-8") as f:
    diag = json.load(f)
print("Skewness original:", {k: round(v, 2) for k, v in diag["skewness_original"].items()})
print("Variables transformadas con log1p:", diag["variables_transformadas"])
mostrar("preprocesamiento_log_transform.png")


`gasto_total` tenía una asimetría extrema (13,01), que tras log1p baja a 1,61; la del salario pasa de
1,06 a −0,24. `ahorro` (skew 0,71) no se transforma: |skew| < 1, contiene valores negativos y es la
variable objetivo del Modelo 1, cuyos coeficientes se interpretan en euros.


## 2. Modelos predictivos principales (rúbrica 4.2)

`modelos_predictivos.py` entrena en una sola ejecución:

- **Modelo 1** — regresión lineal regularizada del ahorro mensual (Ridge o Lasso, elegido por validación cruzada).
- **Modelo 2** — regresión logística del perfil de ahorro (`ahorro_adecuado` / `ahorro_insuficiente`).
- **Modelo avanzado** — ensemble HistGradientBoosting (§3).
- **Modelo 1-bis y capa prescriptiva** — proyección individual y recomendaciones.
- **Modelo 3** — serie temporal del ahorro agregado (§4).

Para evitar la fuga de información, todas las variables predictoras son rezagos de los 3 meses
anteriores (el ahorro del mes t es exactamente salario − gasto del mes t). Cada modelo se compara
con pronósticos ingenuos (baselines) sobre el mismo conjunto de prueba.

> Esta celda es la más costosa del cuaderno (búsqueda de hiperparámetros con GroupKFold sobre
> ~420.000 registros).


In [ ]:
ejecutar_script("modelos_predictivos.py")

In [ ]:
with open(os.path.join(DATA_CLEAN, "metricas_modelos.json"), encoding="utf-8") as f:
    m = json.load(f)
print(m["validacion"])
print(f"Usuarios train/test: {m['usuarios_train']:,} / {m['usuarios_test']:,}")


### 2.1 Modelo 1 — Regresión del ahorro mensual

Ajuste de hiperparámetros (MAE de validación cruzada, media ± desviación entre pliegues):

In [ ]:
cv1 = tabla(m["modelo1_cv_resultados"])
cv1["parametros"] = cv1["parametros"].astype(str)
display(cv1)
print(f"Seleccionado: {m['modelo1_modelo_seleccionado']} {m['modelo1_hiperparametros']}")


Resultados en el conjunto de prueba frente a las baselines:

In [ ]:
res1 = tabla([{"metodo": k, **v} for k, v in m["modelo1_baselines_test"].items()]
             + [{"metodo": f"Modelo 1 ({m['modelo1_modelo_seleccionado']})",
                 "mae": m["modelo1_mae"], "r2": m["modelo1_r2"]}])
display(res1)
mostrar("modelo1_regresion_lineal.png")
mostrar("modelo1_coeficientes.png", 650)


**Lectura.** El modelo mejora con claridad la persistencia y la media de entrenamiento, pero solo un
1,4% el MAE de la media móvil de 3 meses (su R² sí mejora de 0,756 a 0,799): la mayor parte de la
capacidad predictiva procede del propio historial de ahorro. Ridge y Lasso empatan en validación
(la diferencia es menor que la variabilidad entre pliegues).

**Colinealidad.** `salario_lag1_log` y `salario_roll3_log` están correlacionadas al 0,96, y
`salario_roll3_log` recibe un coeficiente muy negativo pese a correlacionar positivamente con el
ahorro. La celda siguiente muestra que la regularización apenas cambia los coeficientes y que estos
son estables entre pliegues: no es inestabilidad numérica, sino un efecto condicional (de supresión)
entre dos variables casi redundantes, por lo que ambas deben interpretarse conjuntamente.


In [ ]:
display(pd.DataFrame(m["modelo1_colinealidad"]["coeficientes"]).T)
print(m["modelo1_nota_colinealidad"])


### 2.2 Modelo 1-bis y capa prescriptiva

In [ ]:
print(f"Usuarios con recomendación de ajuste: {m['prescriptivo_usuarios_con_ajuste']:,} "
      f"({m['prescriptivo_pct_usuarios_con_ajuste']}%)")
print(f"Proporción media de la brecha cubierta: {m['prescriptivo_cobertura_media_brecha_pct']}%")
mostrar("modelo1_explicacion_local_ejemplo.png", 700)
mostrar("modelo1bis_explicacion_usuario_ejemplo.png", 700)
mostrar("prescriptivo_categoria_sugerida.png", 650)
display(pd.read_csv(os.path.join(DATA_CLEAN, "recomendaciones_prescriptivas.csv")).head(10))


La categoría más recomendada es «otros», consecuencia de la categorización incompleta del gasto
en el dataset (salud y educación sin gasto; «otros» con el 68,5% del gasto): es una limitación
documentada en la memoria.


### 2.3 Modelo 2 — Clasificación del perfil de ahorro

Esquema binario por mediana, tras descartar el umbral absoluto, los terciles y tres clases de igual
anchura (historial completo en la memoria). Ajuste de C (F1 macro de validación cruzada):


In [ ]:
cv2 = tabla(m["modelo2_cv_resultados"]); cv2["parametros"] = cv2["parametros"].astype(str)
display(cv2)
res2 = tabla([{"metodo": k, **v} for k, v in m["modelo2_baselines_test"].items()]
             + [{"metodo": f"Modelo 2 (logística, C={m['modelo2_C_seleccionado']})",
                 "accuracy": m["modelo2_accuracy"], "f1_macro": m["modelo2_f1_macro"]}])
display(res2)
mostrar("modelo2_confusion.png", 550)
mostrar("modelo2_coeficientes_por_clase.png", 650)


**Lectura.** El modelo supera a la persistencia de la clase en +0,10 de F1 macro. C apenas influye
(F1 de validación entre 0,7209 y 0,7211): con ~420.000 registros y 18 variables el modelo es
insensible a la regularización. Positivo en el gráfico de coeficientes = mayor probabilidad de
`ahorro_insuficiente`.


## 3. Modelo avanzado — Ensemble (rúbrica 4.3)

HistGradientBoosting (gradient boosting de árboles basado en histogramas) para regresión y
clasificación, con las mismas variables, la misma partición y la misma validación cruzada. Cuantifica
cuánta precisión se sacrifica al exigir un modelo lineal interpretable, y si el techo del Modelo 2 se
debe al algoritmo o a la información disponible.


In [ ]:
ens = m["modelo_avanzado_ensemble"]
for tarea in ["regresion", "clasificacion"]:
    t = tabla(ens[tarea]["cv_resultados"]); t["parametros"] = t["parametros"].astype(str)
    print(f"--- {tarea}: hiperparámetros seleccionados {ens[tarea]['hiperparametros']}")
    display(t)
comp = pd.DataFrame({
    "lineal": [m["modelo1_mae"], m["modelo1_r2"], m["modelo2_accuracy"], m["modelo2_f1_macro"]],
    "ensemble": [ens["regresion"]["mae_test"], ens["regresion"]["r2_test"],
                 ens["clasificacion"]["accuracy_test"], ens["clasificacion"]["f1_macro_test"]]},
    index=["MAE regresión (€)", "R² regresión", "accuracy clasificación", "F1 macro clasificación"])
display(comp)
mostrar("modelo_avanzado_comparacion.png", 950)


**Lectura.** En regresión el ensemble reduce el MAE un 8%: existen relaciones no lineales que el
modelo lineal no capta. En clasificación no mejora nada: el techo es de información, no de
algoritmo. Se mantiene el modelo lineal en la capa prescriptiva porque su descomposición
coeficiente × valor permite justificar cada recomendación ante el usuario.


## 4. Modelo 3 — Serie temporal del ahorro agregado

La media mensual simple está distorsionada por la entrada y salida de usuarios (entre 5.066 y
29.935 activos según el mes). Se construye un **índice encadenado** (variación mes a mes calculada
solo sobre usuarios presentes en ambos meses) y sobre él una regresión con tendencia y
estacionalidad, validada con un **backtest** de 6 meses frente a pronósticos ingenuos.


In [ ]:
mostrar("modelo3_comparacion_naive_vs_encadenado.png", 950)
print(f"R² dentro de muestra: {m['modelo3_r2_insample']} | desviación típica de la serie: "
      f"{m['modelo3_std_serie']} € sobre un nivel de {m['modelo3_nivel_medio_serie']} €")
display(pd.Series(m["modelo3_backtest_mae_6meses"], name="MAE backtest (€)").to_frame())
mostrar("modelo3_proyeccion.png", 950)
display(tabla(m["modelo3_predicciones"]))


**Lectura.** El modelo es peor que la media histórica en el backtest: una vez corregido el efecto de
composición, la serie no tiene tendencia ni estacionalidad que modelar. La aportación de esta sección
es el índice encadenado, que demuestra que las variaciones de la media simple eran un artefacto de
composición de la muestra.


## 5. Análisis no supervisado — Segmentación de usuarios (rúbrica 4.4)

K-Means sobre un perfil agregado **por usuario**, con variables exclusivamente de comportamiento:
tasa de ahorro media (winsorizada en p1–p99), volatilidad del ahorro y composición del gasto por
categoría. Edad, perfil laboral e ingreso se usan solo para describir los segmentos. El número de
segmentos es el máximo global de la silueta en k = 2..8. Los segmentos se relacionan con el TFM
evaluando objetivos de ahorro personalizados por segmento para la capa prescriptiva.


In [ ]:
ejecutar_script("segmentacion_usuarios.py")

In [ ]:
with open(os.path.join(DATA_CLEAN, "segmentacion_diagnostico.json"), encoding="utf-8") as f:
    seg = json.load(f)
print(f"k = {seg['k_elegido']} ({seg['criterio_seleccion_k']}), silueta = {seg['silueta_k_elegido']}")
print(f"NMI con el perfil de origen Tink: {seg['nmi_vs_perfil_origen_tink']} | "
      f"con el perfil laboral: {seg['nmi_vs_perfil_laboral']}")
mostrar("segmentacion_evaluacion_k.png", 950)
cols = ["n_usuarios", "pct_usuarios", "tasa_ahorro_media", "volatilidad_ahorro",
        "pct_gasto_alimentacion", "pct_gasto_ocio", "pct_gasto_transporte", "pct_gasto_otros",
        "ingreso_medio", "n_meses_observados"]
display(pd.DataFrame(seg["resumen_por_cluster"]).set_index("cluster")[cols])
mostrar("segmentacion_clusters_pca.png", 650)
mostrar("segmentacion_perfil_clusters.png", 950)


### Personalización de las recomendaciones por segmento

In [ ]:
display(pd.DataFrame(seg["recomendaciones_por_segmento"]))
mostrar("segmentacion_recomendaciones.png", 900)


**Lectura.**

- El segmento de mayor riesgo (ahorro medio muy negativo y volatilidad alta) está formado por
  usuarios con solo 2 meses de historial: no reciben proyección del Modelo 1-bis, que necesita 3.
  En producción requieren reglas de alerta (arranque en frío).
- Con un objetivo personalizado (mediana del propio segmento), los segmentos de ahorro bajo o
  intermedio pasan a tener brechas alcanzables; el de ahorro máximo, un objetivo más exigente.
- Los segmentos reproducen en buena medida las 8 plantillas del generador sintético (NMI ≈ 0,67)
  aunque el origen no se usó como variable: describen 8 patrones de comportamiento, no la
  diversidad de una población real.


## Conclusiones

| Rúbrica | Evidencia |
|---|---|
| 4.1 Preprocesamiento | One-hot, log1p justificado por asimetría, escalado sin fuga (§1) |
| 4.2 Modelos principales | Modelo 1 (Lasso) y Modelo 2 (logística) con GridSearchCV + GroupKFold y comparación con baselines (§2) |
| 4.3 Modelo avanzado | Ensemble HistGradientBoosting (§3); modelo temporal con backtest (§4) |
| 4.4 No supervisado | K-Means comportamental, k=6, personalización de recomendaciones (§5) |

Los resultados se presentan junto a sus baselines, de modo que se ve qué aporta realmente cada
modelo: el Modelo 2 mejora claramente a los pronósticos ingenuos; el Modelo 1, sobre todo en R²; el
ensemble muestra el margen de mejora no lineal; y el Modelo 3 no supera a la media histórica, aunque
su índice encadenado corrige el efecto de composición de la muestra.
